<a href="https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/simulation/Specification_debt_simulation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""Isolated test of the TLIST -> LIST conversion logic from modelconstruct.clean_expressions."""

import re

class ModelSpecificationError(Exception):
    pass

# --- copy of the inner helpers, lifted out for testing ---

def _convert_tlist_to_list(flat_no_dollar: str) -> str:
    m = re.match(r'^\s*TLIST\b\s*(.*)$', flat_no_dollar, flags=re.IGNORECASE)
    if not m:
        raise ModelSpecificationError(f"Internal: not TLIST: {flat_no_dollar!r}")
    body = m.group(1).strip()

    if '=' not in body:
        raise ModelSpecificationError(f"TLIST missing '=': {flat_no_dollar!r}")
    list_name, rhs = body.split('=', 1)
    list_name = list_name.strip()
    if not list_name:
        raise ModelSpecificationError(f"TLIST empty name: {flat_no_dollar!r}")

    raw_rows = [r.strip() for r in rhs.split('/')]
    raw_rows = [r for r in raw_rows if r != '']
    if len(raw_rows) < 2:
        raise ModelSpecificationError(
            f"TLIST {list_name} needs header + >=1 value row: {flat_no_dollar!r}")

    def tokens(row):
        return [t for t in re.split(r'[\s,]+', row.strip()) if t != '']

    header = tokens(raw_rows[0])
    value_rows = [tokens(r) for r in raw_rows[1:]]

    if not header:
        raise ModelSpecificationError(f"TLIST {list_name} empty header")

    ncols = len(header)
    for k, vr in enumerate(value_rows, 1):
        if len(vr) != ncols:
            raise ModelSpecificationError(
                f"TLIST {list_name}: value row {k} has {len(vr)} entries vs header {ncols}")

    sublists = []
    for j, sublist_name in enumerate(header):
        col = [vr[j] for vr in value_rows]
        sublists.append(f"{sublist_name} : {' '.join(col)}")

    return f"LIST {list_name} = " + ' / '.join(sublists)


def normalize_list_block(block: str) -> str:
    saw_dollar = bool(re.search(r'\$\s*$', block))
    is_tlist   = bool(re.match(r'^\s*TLIST\b', block))

    flat = ' '.join(line.strip() for line in block.splitlines())
    flat = re.sub(r'\s+', ' ', flat).strip()
    flat = re.sub(r'\s*\$\s*$', '', flat)
    flat = re.sub(r'\s*/\s*$', '', flat)

    if is_tlist:
        flat = _convert_tlist_to_list(flat)

    if saw_dollar:
        flat = flat + ' $'

    return flat


# --------------------------------------------------------------------
# tests
# --------------------------------------------------------------------

def show(label, text):
    print(f"--- {label} ---")
    print(text)
    print()


# 1. The exact example from the user's __main__ rewritten as TLIST.
#    Original LIST form (rows = sublists):
#       LIST BANKS = BANKS    : IB      SOREN  MARIE /
#                    COUNTRY  : DENMARK SWEDEN DENMARK /
#                    SELECTED : 1       0      1
#                    $
#    TLIST form (header row = sublist names, columns go DOWN):
tlist1 = """TLIST BANKS = BANKS  COUNTRY  SELECTED /
                       IB     DENMARK  1 /
                       SOREN  SWEDEN   0 /
                       MARIE  DENMARK  1
                       $"""

out1 = normalize_list_block(tlist1)
show("TLIST BANKS -> LIST", out1)
expected1 = "LIST BANKS = BANKS : IB SOREN MARIE / COUNTRY : DENMARK SWEDEN DENMARK / SELECTED : 1 0 1 $"
assert out1 == expected1, f"mismatch:\n got: {out1!r}\nwant: {expected1!r}"
print("OK: matches expected canonical LIST\n")


# 2. TLIST with no trailing $ -- should be preserved (no $ added)
tlist2 = """TLIST SECTORS = SECTORS /
                            NFC /
                            SME /
                            HH"""
out2 = normalize_list_block(tlist2)
show("TLIST SECTORS no-$", out2)
assert not out2.endswith('$')
assert out2 == "LIST SECTORS = SECTORS : NFC SME HH"
print("OK: $-preservation correct\n")


# 3. Single value row (one tuple)
tlist3 = """TLIST X = A B C /
                      1 2 3 $"""
out3 = normalize_list_block(tlist3)
show("TLIST X single value row", out3)
assert out3 == "LIST X = A : 1 / B : 2 / C : 3 $"
print("OK: single-row case\n")


# 4. Comma separators in the rows
tlist4 = """TLIST X = A,B,C /
                      1,2,3 /
                      4,5,6 $"""
out4 = normalize_list_block(tlist4)
show("TLIST X commas", out4)
assert out4 == "LIST X = A : 1 4 / B : 2 5 / C : 3 6 $"
print("OK: comma separators\n")


# 5. Trailing slash on the last value row should be tolerated
tlist5 = """TLIST X = A B /
                      1 2 /
                      3 4 /
                      $"""
out5 = normalize_list_block(tlist5)
show("TLIST X trailing slash", out5)
assert out5 == "LIST X = A : 1 3 / B : 2 4 $"
print("OK: trailing slash tolerated\n")


# 6. Mismatched row width should raise
bad = """TLIST X = A B C /
                   1 2 /
                   3 4 5 $"""
try:
    normalize_list_block(bad)
except ModelSpecificationError as e:
    print(f"OK: bad TLIST raised: {e}\n")
else:
    raise AssertionError("expected ModelSpecificationError for ragged TLIST")


# 7. Non-TLIST (regular LIST) should pass through unchanged in shape
list_normal = """LIST BANKS = BANKS    : IB      SOREN  MARIE /
                              COUNTRY : DENMARK SWEDEN DENMARK  /
                              SELECTED : 1 0 1
                              $"""
out7 = normalize_list_block(list_normal)
show("regular LIST (unchanged shape)", out7)
assert out7.startswith("LIST BANKS")
assert out7.endswith("$")
print("OK: regular LIST untouched\n")

print("ALL TESTS PASSED")

--- TLIST BANKS -> LIST ---
LIST BANKS = BANKS : IB SOREN MARIE / COUNTRY : DENMARK SWEDEN DENMARK / SELECTED : 1 0 1 $

OK: matches expected canonical LIST

--- TLIST SECTORS no-$ ---
LIST SECTORS = SECTORS : NFC SME HH

OK: $-preservation correct

--- TLIST X single value row ---
LIST X = A : 1 / B : 2 / C : 3 $

OK: single-row case

--- TLIST X commas ---
LIST X = A : 1 4 / B : 2 5 / C : 3 6 $

OK: comma separators

--- TLIST X trailing slash ---
LIST X = A : 1 3 / B : 2 4 $

OK: trailing slash tolerated

OK: bad TLIST raised: TLIST X: value row 1 has 2 entries vs header 3

--- regular LIST (unchanged shape) ---
LIST BANKS = BANKS : IB SOREN MARIE / COUNTRY : DENMARK SWEDEN DENMARK / SELECTED : 1 0 1 $

OK: regular LIST untouched

ALL TESTS PASSED
